# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
import os, sys, subprocess

# Run the import check in a throwaway subprocess, not directly in this kernel,
# so any crash there can't take down the kernel itself.
def _check_import(modname, timeout=60):
    r = subprocess.run([sys.executable, '-c', f'import {modname}'],
                        capture_output=True, text=True, timeout=timeout)
    return r.returncode == 0, r

def _pip_reinstall(pkg, timeout=180):
    return subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--force-reinstall',
         '--no-deps', '--no-cache-dir', pkg],
        capture_output=True, text=True, timeout=timeout,
    )

ok, r = _check_import('mani_skill.utils')
if not ok:
    print(f'mani_skill broken (exit {r.returncode}) — diagnosing...')
    print(r.stderr[-1500:])
    # Real root cause found previously: mani_skill.utils.common does
    # `import sapien.physx`, and sapien (mani_skill's physics/rendering engine
    # dependency) can go missing independently of mani_skill itself. A --no-deps
    # reinstall of ONLY mani-skill never fixes that — sapien must be reinstalled
    # too. Reinstalling both individually (still --no-deps each) avoids dragging
    # in the full dependency tree (torch etc.), which is what made earlier
    # attempts slow.
    for pkg in ['sapien', 'mani-skill']:
        print(f'Reinstalling {pkg}...')
        try:
            rr = _pip_reinstall(pkg)
        except subprocess.TimeoutExpired:
            raise RuntimeError(f'{pkg} reinstall timed out after 180s — network issue reaching PyPI.')
        if rr.returncode != 0:
            print(f'{pkg} reinstall FAILED — pip output:')
            print(rr.stdout[-2000:])
            print(rr.stderr[-2000:])
            raise RuntimeError(f'{pkg} reinstall failed — see pip output above')
        print(f'{pkg} reinstalled OK.')

    ok2, r2 = _check_import('mani_skill.utils')
    if not ok2:
        print(f'Still broken after reinstalling sapien + mani-skill (exit {r2.returncode}):')
        print(r2.stderr[-3000:])
        raise RuntimeError(
            'mani_skill still broken after reinstalling sapien + mani-skill. '
            'Try: Runtime > Disconnect and delete runtime, then retry on a fresh VM.'
        )
    print('mani_skill verified OK. Continuing...')

if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')
    os.system('apt-get install -qq libvulkan1 libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')
    os.system('pip install -q einops timm pyyaml mani-skill accelerate')
    os.system('pip install -q "numpy<2"')
    os.system('pip install -q "protobuf>=5.28.0"')
    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.chdir("/content/rdt-igtesting")
os.system("git pull -q")

TASKS = ["PickCube-v1", "StackCube-v1", "PushCube-v1", "PegInsertionSide-v1", "PlugCharger-v1"]
for task in TASKS:
    shutil.copy(
        hf_hub_download("robotics-diffusion-transformer/maniskill-model",
                        "lang_embeds/text_embed_" + task + ".pt"),
        "lang_embed_" + task + ".pt"
    )
    print("Downloaded lang embed:", task)

shutil.copy("lang_embed_PickCube-v1.pt", "lang_embed.pt")
print("Ready.")

In [ ]:
# @title Rollout settings
n_successes = 1  # @param {type:"integer"}
task = "PickCube-v1"  # @param ["PickCube-v1", "StackCube-v1", "PushCube-v1", "PegInsertionSide-v1", "PlugCharger-v1"]

from run_rollout_cell import run
run(task=task, n=n_successes)

In [ ]:
import os
os.environ["SKIP_IG"] = "1"
os.environ["MANISKILL_TASK"] = task
os.environ["LANG_EMBED"] = "lang_embed_" + task + ".pt"
%run -i rdt_blurig.py
%run -i run_ig.py